In [ ]:
import pandas as pd
import json
import sqlite3
import re

# 1. Carregar o JSON (O Ouro Completo)
caminho_do_json = r"C:\Users\trafe\OneDrive\Documentos\documento\CCO - Luiz\CCO_Relatorios\qualificacao_2026-05-14.json"

# SUPERPODER 1: Achar a data automaticamente pelo nome do arquivo
data_arquivo_match = re.search(r'\d{4}-\d{2}-\d{2}', caminho_do_json)
data_operacao = data_arquivo_match.group(0) if data_arquivo_match else '1900-01-01'

with open(caminho_do_json, 'r', encoding='utf-8') as f:
    dados_json = json.load(f)

# 2. O Achatamento Mágico (agora puxando os dados novos do HTML)
# O errors='ignore' protege o código caso um carro não tenha o campo 'ocorrencias' preenchido
tabela_cco = pd.json_normalize(
    dados_json,
    record_path='viagens',
    meta=['carro', 'linha', 'turno', 'servico', 'motorista', 'pontoInicial', 'pontoFinal', 'garagem'],
    errors='ignore'
)

# SUPERPODER 2: Se tiver uma lista de ocorrências, transforma em texto para o SQL engolir
if 'ocorrencias' in tabela_cco.columns:
    tabela_cco['ocorrencias'] = tabela_cco['ocorrencias'].astype(str)
else:
    tabela_cco['ocorrencias'] = "Sem Ocorrências"

# 3. Construir a "Estante" nova (O Cofre v2)
conexao = sqlite3.connect('banco_cco_qualificacao.db')
cursor = conexao.cursor()

cursor.execute("CREATE TABLE IF NOT EXISTS dim_carros (prefixo TEXT PRIMARY KEY)")
cursor.execute("CREATE TABLE IF NOT EXISTS dim_linhas (codigo TEXT PRIMARY KEY)")
cursor.execute("CREATE TABLE IF NOT EXISTS dim_motoristas (nome_matricula TEXT PRIMARY KEY)")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fato_viagens (
    id_viagem INTEGER PRIMARY KEY AUTOINCREMENT,
    data_operacao TEXT,
    prefixo_carro TEXT,
    codigo_linha TEXT,
    motorista TEXT,
    turno TEXT,
    servico TEXT,
    sentido TEXT,
    ponto_inicial TEXT,
    ponto_final TEXT,
    garagem TEXT,
    prog_saida TEXT,
    prog_chegada TEXT,
    real_saida TEXT,
    real_chegada TEXT,
    ocorrencias TEXT
)
""")

# 4. A Vassoura (Drop & Reload automático para a data do arquivo)
cursor.execute("DELETE FROM fato_viagens WHERE data_operacao = ?", (data_operacao,))
conexao.commit()
print(f"🧹 Faxina feita! Nenhuma viagem duplicada do dia {data_operacao}.")

# 5. Povoar as Dimensões (Cadastros)
for carro in tabela_cco['carro'].dropna().unique():
    cursor.execute("INSERT OR IGNORE INTO dim_carros (prefixo) VALUES (?)", (carro,))

for linha in tabela_cco['linha'].dropna().unique():
    cursor.execute("INSERT OR IGNORE INTO dim_linhas (codigo) VALUES (?)", (linha,))

# Tratar motoristas para não dar erro se vier vazio
if 'motorista' in tabela_cco.columns:
    for mot in tabela_cco['motorista'].dropna().unique():
        cursor.execute("INSERT OR IGNORE INTO dim_motoristas (nome_matricula) VALUES (?)", (mot,))

conexao.commit()

# 6. Preparar a Tabela Fato
fato_pandas = tabela_cco.rename(columns={
    'carro': 'prefixo_carro',
    'linha': 'codigo_linha',
    'tipo': 'sentido',
    'pontoInicial': 'ponto_inicial',
    'pontoFinal': 'ponto_final',
    'progSaida': 'prog_saida',
    'progChegada': 'prog_chegada',
    'realSaida': 'real_saida',
    'realChegada': 'real_chegada'
}).copy()

fato_pandas['data_operacao'] = data_operacao

# Preencher colunas que possam não existir em algumas exportações
for col in ['motorista', 'garagem', 'ponto_inicial', 'ponto_final']:
    if col not in fato_pandas.columns:
        fato_pandas[col] = "Não Informado"

# Alinhar a ordem exata das gavetas do SQL
colunas_fato = ['data_operacao', 'prefixo_carro', 'codigo_linha', 'motorista', 'turno', 'servico', 
                'sentido', 'ponto_inicial', 'ponto_final', 'garagem', 'prog_saida', 'prog_chegada', 
                'real_saida', 'real_chegada', 'ocorrencias']

fato_pandas = fato_pandas[colunas_fato]

# 7. A Grande Injeção
fato_pandas.to_sql('fato_viagens', conexao, if_exists='append', index=False)
conexao.close()

print(f"✅ SUCESSO TOTAL! {len(fato_pandas)} viagens qualificadas do dia {data_operacao} estão no cofre v2.0!")

🧹 Faxina feita! Nenhuma viagem duplicada do dia 2026-05-14.


DatabaseError: Execution failed